# Sentinel-2 NDVI Change Detection: Phenikaa Hospital Area

This notebook regenerates `public/sample-analysis/phenikaa-area-ndvi-change.json` from Google Earth Engine. Run all cells from top to bottom after authenticating Earth Engine.

In [ ]:
import json
from pathlib import Path

import ee
import geemap

PROJECT_ID = "project-gis-499303"
DATASET = "COPERNICUS/S2_SR_HARMONIZED"
OUTPUT_RELATIVE_PATH = Path("public/sample-analysis/phenikaa-area-ndvi-change.json")

ee.Initialize(project=PROJECT_ID)

## Area of interest and analysis periods

The same October-to-April seasonal window is used for both periods to reduce seasonal vegetation differences.

In [ ]:
center_lon = 105.749433
center_lat = 21.039419

aoi = ee.Geometry.Polygon([
    [
        [105.7350, 21.0259],
        [105.7639, 21.0259],
        [105.7639, 21.0529],
        [105.7350, 21.0529],
        [105.7350, 21.0259],
    ]
])

before_start = "2018-10-01"
before_end = "2019-04-30"
after_start = "2024-10-01"
after_end = "2025-04-30"

Map = geemap.Map(center=[center_lat, center_lon], zoom=14)
Map.addLayer(aoi, {"color": "red"}, "AOI - Phenikaa Hospital Area")
Map

## Sentinel-2 cloud masking and median composites

Scenes are filtered to less than 40% reported cloud cover. The SCL mask keeps vegetation, non-vegetated surfaces, water, unclassified pixels, and snow/ice; cloud and cloud-shadow classes are excluded.

In [ ]:
def mask_s2_clouds(image):
    scl = image.select("SCL")
    valid_mask = (
        scl.eq(4)
        .Or(scl.eq(5))
        .Or(scl.eq(6))
        .Or(scl.eq(7))
        .Or(scl.eq(11))
    )
    return image.updateMask(valid_mask).divide(10000)


def get_s2_composite(start_date, end_date, geometry):
    collection = (
        ee.ImageCollection(DATASET)
        .filterBounds(geometry)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
        .map(mask_s2_clouds)
    )
    return collection.median().clip(geometry), collection.size()


before_img, before_count = get_s2_composite(before_start, before_end, aoi)
after_img, after_count = get_s2_composite(after_start, after_end, aoi)

before_count_value = int(before_count.getInfo())
after_count_value = int(after_count.getInfo())
print("Before image count:", before_count_value)
print("After image count:", after_count_value)

## NDVI and NDVI difference

`NDVI = (B8 - B4) / (B8 + B4)`. The change image is after NDVI minus before NDVI.

In [ ]:
ndvi_before = before_img.normalizedDifference(["B8", "B4"]).rename("NDVI")
ndvi_after = after_img.normalizedDifference(["B8", "B4"]).rename("NDVI")
ndvi_diff = ndvi_after.subtract(ndvi_before).rename("NDVI_Diff")

Map.addLayer(ndvi_before, {"min": -0.2, "max": 0.8, "palette": ["brown", "yellow", "green"]}, "NDVI before")
Map.addLayer(ndvi_after, {"min": -0.2, "max": 0.8, "palette": ["brown", "yellow", "green"]}, "NDVI after")
Map.addLayer(ndvi_diff, {"min": -0.5, "max": 0.5, "palette": ["red", "white", "blue"]}, "NDVI difference")
Map

## Difference statistics and change thresholds

Vegetation loss and gain are pixels outside the AOI-wide NDVI-difference mean plus or minus 1.5 standard deviations.

In [ ]:
difference_stats = ndvi_diff.reduceRegion(
    reducer=ee.Reducer.mean().combine(
        reducer2=ee.Reducer.stdDev(),
        sharedInputs=True,
    ),
    geometry=aoi,
    scale=10,
    maxPixels=1_000_000_000,
)

difference_mean = ee.Number(difference_stats.get("NDVI_Diff_mean"))
difference_std_dev = ee.Number(difference_stats.get("NDVI_Diff_stdDev"))
loss_threshold = difference_mean.subtract(difference_std_dev.multiply(1.5))
gain_threshold = difference_mean.add(difference_std_dev.multiply(1.5))

difference_mean_value = float(difference_mean.getInfo())
difference_std_dev_value = float(difference_std_dev.getInfo())
loss_threshold_value = float(loss_threshold.getInfo())
gain_threshold_value = float(gain_threshold.getInfo())

print("NDVI difference mean:", difference_mean_value)
print("NDVI difference standard deviation:", difference_std_dev_value)
print("Vegetation loss threshold:", loss_threshold_value)
print("Vegetation gain threshold:", gain_threshold_value)

## Vegetation loss and gain area

Earth Engine pixel area is masked by each change class, summed at 10 m scale, and converted from square metres to square kilometres.

In [ ]:
vegetation_loss = ndvi_diff.lt(loss_threshold).selfMask()
vegetation_gain = ndvi_diff.gt(gain_threshold).selfMask()
pixel_area_km2 = ee.Image.pixelArea().divide(1_000_000).rename("area_km2")


def masked_area_km2(mask):
    area_stats = pixel_area_km2.updateMask(mask).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=aoi,
        scale=10,
        maxPixels=1_000_000_000,
    )
    return ee.Number(area_stats.get("area_km2"))


vegetation_loss_km2 = masked_area_km2(vegetation_loss)
vegetation_gain_km2 = masked_area_km2(vegetation_gain)
vegetation_loss_km2_value = float(vegetation_loss_km2.getInfo())
vegetation_gain_km2_value = float(vegetation_gain_km2.getInfo())

print("Vegetation loss area (km2):", vegetation_loss_km2_value)
print("Vegetation gain area (km2):", vegetation_gain_km2_value)

Map.addLayer(vegetation_loss, {"palette": ["red"]}, "Vegetation loss")
Map.addLayer(vegetation_gain, {"palette": ["blue"]}, "Vegetation gain")
Map

## Export dashboard JSON

The output dictionary matches the schema used by `public/sample-analysis/phenikaa-area-ndvi-change.json`. The repository root is located from the current working directory so the export works when Jupyter starts either at the repository root or inside `notebooks/`.

In [ ]:
analysis_result = {
    "aoiName": "Phenikaa Hospital Area, Hanoi",
    "aoiDescription": "Phu Dien - Xuan Phuong area near Phenikaa University Hospital, Hanoi",
    "center": {
        "longitude": center_lon,
        "latitude": center_lat,
    },
    "dataset": DATASET,
    "index": "NDVI",
    "beforeRange": [before_start, before_end],
    "afterRange": [after_start, after_end],
    "method": {
        "comparison": "same-season comparison",
        "seasonWindow": "October to April",
        "composite": "median composite",
        "threshold": "mean \u00b1 1.5 standard deviation of NDVI difference",
        "lossRule": "NDVI_Diff < mean - 1.5 * stdDev",
        "gainRule": "NDVI_Diff > mean + 1.5 * stdDev",
    },
    "imageCounts": {
        "before": before_count_value,
        "after": after_count_value,
    },
    "stats": {
        "differenceMean": difference_mean_value,
        "differenceStdDev": difference_std_dev_value,
        "vegetationLossThreshold": loss_threshold_value,
        "vegetationGainThreshold": gain_threshold_value,
        "vegetationLossKm2": vegetation_loss_km2_value,
        "vegetationGainKm2": vegetation_gain_km2_value,
    },
}

search_paths = [Path.cwd(), *Path.cwd().parents]
repo_root = next((path for path in search_paths if (path / "PROJECT_PLAN.md").exists()), None)
if repo_root is None:
    raise FileNotFoundError("Could not locate the repository root containing PROJECT_PLAN.md")

output_path = repo_root / OUTPUT_RELATIVE_PATH
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as output_file:
    json.dump(analysis_result, output_file, ensure_ascii=False, indent=2)
    output_file.write("\n")

print(f"Wrote {output_path}")
analysis_result